In [39]:
import sqlparse
from sqlparse.sql import IdentifierList, Identifier
from sqlparse.tokens import Keyword, DML
import os
import pandas as pd
from collections import defaultdict

from IPython.display import HTML
from IPython.display import display_html


%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
from utils.utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
import networkx as nx
from networkx.readwrite import json_graph
import pickle

def load_sql_rep(fn, dummy=None):
    assert ".pkl" in fn
    try:
        with open(fn, "rb") as f:
            query = pickle.load(f)
    except Exception as e:
        print(e)
        print(fn + " failed to load...")
        exit(-1)

    query["subset_graph"] = nx.OrderedDiGraph(json_graph.adjacency_graph(query["subset_graph"]))
    query["join_graph"] = json_graph.adjacency_graph(query["join_graph"])
    if "subset_graph_paths" in query:
        query["subset_graph_paths"] = nx.OrderedDiGraph(json_graph.adjacency_graph(query["subset_graph_paths"]))

    return query

In [41]:
SQL_DIR = "multi_column_1/"

In [42]:
WK = "ceb"

# QDIR = "./imdb-sqls/" + SQL_DIR
# OUTPATH = os.path.join("./imdb-dfs/" + SQL_DIR)

QDIR = "./imdb-new-sqls/" + SQL_DIR
OUTPATH = os.path.join("./imdb-new-dfs/" + SQL_DIR)

make_dir(OUTPATH)

COLS_TO_TABLES = {}


In [43]:
qdict = {}
qfns = []

# QREP_DIR = './imdb-workload/' + SQL_DIR
QREP_DIR = './imdb-new-workload/' + SQL_DIR


if WK in ["job", "ceb"]:
    qfns = os.listdir(QREP_DIR)
    for fn in qfns:
        qfn = os.path.join(QREP_DIR, fn)
        if ".pkl" in qfn:
            qdict[fn] = load_sql_rep(qfn)

In [44]:
FNS = os.listdir(QDIR)

In [45]:
sqls = []
qreps = []

for fi, fn in enumerate(FNS):
    if ".sql" not in fn:
        continue
    if "all" in fn:
        continue
    
    fnkey = fn.replace(".sql", ".pkl")
    qreps.append(qdict[fnkey])
        
    fn = os.path.join(QDIR, fn)
    with open(fn, "r", encoding='utf-8') as f:
        sql = f.read()

    sql = sql.replace("IN", "in")
    sql = sql.replace("In", "in")
    sql = sql.replace("ILIKE", "like")
    sql = sql.replace("LIKE", "like")
    
    if "return_rank" in sql:
        print(sql)
    sqls.append(sql)

In [46]:
print(len(sqls), len(qreps))

1 1


In [47]:
for i, sql in enumerate(sqls):
    sql = sql.replace("IN", "in")
    sql = sql.replace("In", "in")
    sql = sql.replace("ILIKE", "like")
    sql = sql.replace("LIKE", "like")
    sqls[i] = sql

In [48]:
len(sqls)

1

In [49]:
#opdf,exprdf = parse_sqls_single([sql], 0, COLS_TO_TABLES)
#opdf
parse_sql_preds(sql)

,op,dtype,optype,constant,input,column,opstring,is_id
0,=,string,in,'4',it1,id,it1.id in ('4'),0
1,=,string,in,'Danish',mi1,info,"mi1.info in ('Danish','Spanish')",0
2,=,string,in,'Spanish',mi1,info,"mi1.info in ('Danish','Spanish')",0
3,LIKE,unknown,contains,'%lise%',n,name,n.name like '%lise%',0
4,=,string,in,'tv series',kt,kind,"kt.kind in ('tv series','video game','video mo...",0
5,=,string,in,'video game',kt,kind,"kt.kind in ('tv series','video game','video mo...",0
6,=,string,in,'video movie',kt,kind,"kt.kind in ('tv series','video game','video mo...",0
7,=,string,in,'director',rt,role,"rt.role in ('director','editor','miscellaneous...",0
8,=,string,in,'editor',rt,role,"rt.role in ('director','editor','miscellaneous...",0
9,=,string,in,'miscellaneous crew',rt,role,"rt.role in ('director','editor','miscellaneous...",0


In [50]:
start = time.time()
opdf,exprdf = parse_sqls_par(sqls, COLS_TO_TABLES)
print("parse sqls took(s): ", int(time.time()-start))
print("Num Ops: ", len(opdf), "Num Expressions: ", len(exprdf))

number of sql groups:  1
parse sqls took(s):  4
Num Ops:  12 Num Expressions:  5


In [51]:
print("len(qreps) is ",len(qreps))
if len(qreps) != 0:
#     for jid in set(exprdf["jobid"]):
#         qrep = qreps[int(jid)]
#         sql = sqls[int(jid)]
#         assert qrep["sql"].lower() == sql.lower()
#         tmp = exprdf[exprdf.jobid == jid]

    rowcounts = []
    totals = []
    sels = []
    filtersqls = []
    
    for _,row in exprdf.iterrows():
        jid = row["jobid"]
        qrep = qreps[int(jid)]
        sql = sqls[int(jid)]
        try:
            cards = qrep["subset_graph"].nodes()[tuple([row["alias"],])]["cardinality"]
            rowcounts.append(cards["actual"])
            totals.append(cards["total"])
            sels.append(min(float(cards["actual"]) / cards["total"], 1.0))
            
            filtersql = row["filtersql"]
            real_name = qrep["join_graph"].nodes()[row["alias"]]["real_name"]
            filtersql = filtersql.replace(row["input"], real_name, 1)
            filtersqls.append(filtersql)
        except:
            rowcounts.append(-1)
            totals.append(-1)
            sels.append(-1)
            filtersqls.append(-1)
        

len(qreps) is  1


In [52]:
if len(qreps) == 0:
    exprdf["InputCardinality"] = -1
    exprdf["RowCount"] = -1
    exprdf["Selectivity"] = -1
else:
    exprdf["InputCardinality"] = totals
    exprdf["Selectivity"] = sels
    exprdf["RowCount"] = rowcounts
    exprdf["filtersql"] = filtersqls

In [53]:
print(len(set(opdf["exprhash"])))
opdf.head(5)

5


,op,dtype,optype,constant,input,column,opstring,is_id,jobid,discrete_ops,cont_ops,like_ops,alias,exprhash
0,=,string,in,'4',it,id,it1.id in ('4'),0,0,1,0,0,it1,201247606223691005679595717410545984913115377236
1,=,string,in,'Danish',mi,info,"mi1.info in ('Danish','Spanish')",0,0,1,0,0,mi1,915029170915645337045155848430817461122198949223
2,=,string,in,'Spanish',mi,info,"mi1.info in ('Danish','Spanish')",0,0,1,0,0,mi1,915029170915645337045155848430817461122198949223
3,LIKE,unknown,contains,'%lise%',n,name,n.name like '%lise%',0,0,0,0,1,n,1412706111350165054529954383171903240337882399016
4,=,string,in,'tv series',kt,kind,"kt.kind in ('tv series','video game','video mo...",0,0,1,0,0,kt,1333422666374384445257481059356447143403429617557


In [54]:
print(len(set(exprdf["exprhash"])))
exprdf.head(5)

5


,filtersql,jobid,input,alias,exprhash,unique_filter_cols,discrete_ops,cont_ops,like_ops,is_id,num_unique_ops,num_ops,num_discrete_consts,num_likes,like_lens,InputCardinality,Selectivity,RowCount
0,SELECT COUNT(*) from movie_info AS mi1 WHERE m...,0,mi,mi1,915029170915645337045155848430817461122198949223,1,1,0,0,0,1,2,2,0,0,6181550,0.015893,98243
1,SELECT COUNT(*) from role_type AS rt WHERE rt....,0,rt,rt,234388626842304026844118353514313021009107246290,1,1,0,0,0,1,5,5,0,0,12,0.416667,5
2,SELECT COUNT(*) from kind_type AS kt WHERE kt....,0,kt,kt,1333422666374384445257481059356447143403429617557,1,1,0,0,0,1,3,3,0,0,7,0.428571,3
3,SELECT COUNT(*) from info_type AS it1 WHERE it...,0,it,it1,201247606223691005679595717410545984913115377236,1,1,0,0,0,1,1,1,0,0,113,0.008850,1
4,SELECT COUNT(*) from name AS n WHERE n.name li...,0,n,n,1412706111350165054529954383171903240337882399016,1,0,0,1,0,1,1,0,1,8,1736455,0.001572,2729


In [55]:
print(len(exprdf))
print(len(set(exprdf["jobid"])))

5
1


In [56]:
print(len(sqls))

1


In [57]:
exprdf["Selectivity"].describe()

count    5.000000
mean     0.174310
std      0.226769
min      0.001572
25%      0.008850
50%      0.015893
75%      0.416667
max      0.428571
Name: Selectivity, dtype: float64

In [58]:
print(len(opdf))
edf = exprdf[["exprhash", "InputCardinality", "Selectivity", "RowCount"]]
opdf = opdf.merge(edf, on="exprhash", how="inner")
print(len(opdf))
opdf.head(5)

12
12


,op,dtype,optype,constant,input,column,opstring,is_id,jobid,discrete_ops,cont_ops,like_ops,alias,exprhash,InputCardinality,Selectivity,RowCount
0,=,string,in,'4',it,id,it1.id in ('4'),0,0,1,0,0,it1,201247606223691005679595717410545984913115377236,113,0.008850,1
1,=,string,in,'Danish',mi,info,"mi1.info in ('Danish','Spanish')",0,0,1,0,0,mi1,915029170915645337045155848430817461122198949223,6181550,0.015893,98243
2,=,string,in,'Spanish',mi,info,"mi1.info in ('Danish','Spanish')",0,0,1,0,0,mi1,915029170915645337045155848430817461122198949223,6181550,0.015893,98243
3,LIKE,unknown,contains,'%lise%',n,name,n.name like '%lise%',0,0,0,0,1,n,1412706111350165054529954383171903240337882399016,1736455,0.001572,2729
4,=,string,in,'tv series',kt,kind,"kt.kind in ('tv series','video game','video mo...",0,0,1,0,0,kt,1333422666374384445257481059356447143403429617557,7,0.428571,3


In [59]:
exprdf["filtersql"].values

array(["SELECT COUNT(*) from movie_info AS mi1 WHERE mi1.info in ('Danish','Spanish')",
       "SELECT COUNT(*) from role_type AS rt WHERE rt.role in ('director','editor','miscellaneous crew','producer','production designer')",
       "SELECT COUNT(*) from kind_type AS kt WHERE kt.kind in ('tv series','video game','video movie')",
       "SELECT COUNT(*) from info_type AS it1 WHERE it1.id in ('4')",
       "SELECT COUNT(*) from name AS n WHERE n.name like '%lise%'"],
      dtype=object)

In [60]:
if "stack" in WK:
    opdf["opstring"] = -1

In [61]:
exprdf.head(5)

,filtersql,jobid,input,alias,exprhash,unique_filter_cols,discrete_ops,cont_ops,like_ops,is_id,num_unique_ops,num_ops,num_discrete_consts,num_likes,like_lens,InputCardinality,Selectivity,RowCount
0,SELECT COUNT(*) from movie_info AS mi1 WHERE m...,0,mi,mi1,915029170915645337045155848430817461122198949223,1,1,0,0,0,1,2,2,0,0,6181550,0.015893,98243
1,SELECT COUNT(*) from role_type AS rt WHERE rt....,0,rt,rt,234388626842304026844118353514313021009107246290,1,1,0,0,0,1,5,5,0,0,12,0.416667,5
2,SELECT COUNT(*) from kind_type AS kt WHERE kt....,0,kt,kt,1333422666374384445257481059356447143403429617557,1,1,0,0,0,1,3,3,0,0,7,0.428571,3
3,SELECT COUNT(*) from info_type AS it1 WHERE it...,0,it,it1,201247606223691005679595717410545984913115377236,1,1,0,0,0,1,1,1,0,0,113,0.008850,1
4,SELECT COUNT(*) from name AS n WHERE n.name li...,0,n,n,1412706111350165054529954383171903240337882399016,1,0,0,1,0,1,1,0,1,8,1736455,0.001572,2729


In [62]:
print(opdf["is_id"].mean())
print(exprdf["is_id"].mean())

0.0
0.0


In [63]:
set(opdf["column"])

{'id', 'info', 'kind', 'name', 'role'}

In [64]:
opdf["workload"] = WK
exprdf["workload"] = WK
opdf["db"] = WK
exprdf["db"] = WK

import time
start = time.time()

opfn = os.path.join(OUTPATH, "op_df.csv")
exprfn = os.path.join(OUTPATH, "expr_df.csv")

opdf.to_csv(opfn, index=False)
print("opdf to csv took: ", time.time()-start)

exprdf.to_csv(exprfn, index=False)

opdf to csv took:  0.002500295639038086


In [65]:
print(opfn)

./imdb-new-dfs/multi_column_1/op_df.csv
